# What are the metabolites availble to STM during infection and how do they affect growth? 

```
see Obsidian Notes:
GEMs Meeting 07.23.26 Agenda,
Oxygen Gradient Study Analysis,
07.22.26
```
## Background 
We don't know all the metabolites present and available to the microbiome community in the gut. We know of some metabolites in the gut - for example, microbe-derived succinate, host-derived glucose, and microbe-derived 1,2-propanediol. 

The objective of this study is to see what metabolites cause a difference in growth betweeen the stm_only and mixed_stm models and within the models. 

What are the metabolites available to STM during infection that differentiate growth in the stm_only versus stm_mixed communities?

**This is a two part question:** 
1) What are the metabolites available to STM during infection?
2) What is effect of each metabolite on stm_only and mixed_stm growth?

**Approach:**
- I will add one exchange reaction from the Stm_v1_0 model to the infection medium and simulate growth in the stm_only and mixed_stm model.
- I will do this the entire list of exchange reactions to see how presence of each metabolite affects growth in both constrained models. 

This approach is dependent on a defined infection medium. To evaluate the difference in growth within models I need to have a control growth rate for each. To simulate a control growth rate I need to have a control infection medium. The control infection medium should have only metabolites we can confirm are present in the gut during infection. 

Given that I am comparing between the stm_only model and the mixed_stm model, the infection medium should only have host-derived metabolites because I want to know how microbe-derived metabolites affect growth between the stm_only and mixed_stm model. *Input from SW here*

07/29/29

On a related note, I don't remember the rationale for designing a gut infection medium and not a basal gut medium? *Input from SW*

## Defining a Control Infection Medium
07/28/26

The first step is to define a control infection medium of host-derived metabolites available in the gut during infection. I'm going to make this infection medium step-by-step and cite why I think the metabolite should be present. I'll also add papers that I come across that are either about simulating gut conditions or have methods for defining medium conditions.  

08/03/26

I'm just going to start writing the code for this experiment... 

# Infection Medium Nutrient Sweep Study

```
Continuing work from: 
~/2026_GEMs/stm_model_test/062226_062626.ipynb
```

## Purpose
Create a nutrient gradient for each exchange reaction in the infection medium, run FBA, and evaluate growth results to create a defined, basal infection medium. 

### Experimental Design

| **Sample** | **iMAT Threshold** | **Model Name** |
|:---|---|:---|
| 278 | 10%-80%-10% | mixed_stm1090_model |
| 278 | 15%-70%-15% | mixed_stm1585_model |
| 278 | 25%-50%-25% | mixed_stm2575_model |
| 279 | 10%-80%-10% | stm1090_model |
| 279 | 15%-70%-15% | stm1585_model |
| 279 | 25%-50%-25% | stm2575_model |
|STM_V1_0 | None | control_model |

## Generate Models
### Prepare the Gene Weights

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [2]:
# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

In [3]:
# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes

print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671


In [4]:
# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [5]:
# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

STM_V1_0 Control Model has 1271 total genes


In [6]:
# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

In [7]:
# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

All models have 1271 genes


### Convert Gene Weights to Reaction Weights

In [8]:
# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

All models have 2545 reactions


### Generate Constrained Models with iMAT

In [9]:
# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmp7ug5w5vz.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpsbdxeek6.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpsha5gvv1.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp4bbp_sb0.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpx_gar6z0.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpoggc_d1z.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [10]:
# Make a dictionary of the models 

original_models = {
    "stm_only_1090": stm1090_model,
    "stm_only_1585": stm1585_model,
    "stm_only_2575": stm2575_model,
    "mixed_stm_1090": mixedstm1090_model,
    "mixed_stm_1585": mixedstm1585_model,
    "mixed_stm_2575": mixedstm2575_model
}

In [23]:
fba_models = {
    model_name: model.copy()
    for model_name, model in original_models.items()
}

Read LP format model from file /tmp/tmp8cz41yb4.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpb94ffr2i.lp
Reading time = 0.02 seconds
: 3201 rows, 6460 columns, 25176 nonzeros
Read LP format model from file /tmp/tmpqhw1yaj9.lp
Reading time = 0.04 seconds
: 3949 rows, 7128 columns, 28088 nonzeros
Read LP format model from file /tmp/tmptv620bs6.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp4zl0w9pn.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmph0hnbqby.lp
Reading time = 0.03 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


## Experimental Medium Conditions

08/04/26

1) An infection medium with all known metabolites during infection
2) An infection medium with all known metabolites during infection minus microbe-derived metabolites


### Default Medium

In [11]:
# List the default medium exchange reactions, metabolites, and uptake value

for rxn_id, uptake_value in control_model.medium.items():
    rxn = control_model.reactions.get_by_id(rxn_id)
    met = list(rxn.metabolites)[0]
    print(rxn_id, "|", met.id, "|", met.name, "|", uptake_value)

EX_ca2_e | ca2_e | Calcium | 1000.0
EX_cbl1_e | cbl1_e | Cob(I)alamin | 0.01
EX_cl_e | cl_e | Chloride | 1000.0
EX_co2_e | co2_e | CO2 CO2 | 1000.0
EX_cobalt2_e | cobalt2_e | Co2+ | 1000.0
EX_cu2_e | cu2_e | Copper | 1000.0
EX_mg2_e | mg2_e | Magnesium | 1000.0
EX_fe2_e | fe2_e | Fe2+ mitochondria | 1000.0
EX_fe3_e | fe3_e | Iron (Fe3+) | 1000.0
EX_mn2_e | mn2_e | Manganese | 1000.0
EX_mobd_e | mobd_e | Molybdate | 1000.0
EX_na1_e | na1_e | Sodium | 1000.0
EX_nh4_e | nh4_e | Ammonium | 1000.0
EX_o2_e | o2_e | O2 O2 | 18.5
EX_pi_e | pi_e | Phosphate | 1000.0
EX_zn2_e | zn2_e | Zinc | 1000.0
EX_so4_e | so4_e | Sulfate | 1000.0
EX_glc__D_e | glc__D_e | D-Glucose | 5.0
EX_tungs_e | tungs_e | Tungstate | 1000.0
EX_h2o_e | h2o_e | H2O H2O | 100.0
EX_h_e | h_e | H+ | 100.0
EX_k_e | k_e | Potassium | 1000.0


In [12]:
# Create a default_medium dictionary

default_medium = {
 'EX_ca2_e': 1000.0,
 'EX_cbl1_e': 0.01,
 'EX_cl_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_o2_e': 18.5,
 'EX_pi_e': 1000.0,
 'EX_zn2_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_glc__D_e': 5.0,
 'EX_tungs_e': 1000.0,
 'EX_h2o_e': 100.0,
 'EX_h_e': 100.0,
 'EX_k_e': 1000.0}

### Infection Medium

In [13]:
# Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

anaerobic_infection_medium = dict(default_medium)
anaerobic_infection_medium.update({
    "EX_o2_e": 0.0,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [14]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

A_infection_medium = dict(default_medium)
A_infection_medium.update({
    "EX_o2_e": 0.5,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [15]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

B_infection_medium = dict(default_medium)
B_infection_medium.update({
    "EX_o2_e": 1,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [16]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

C_infection_medium = dict(default_medium)
C_infection_medium.update({
    "EX_o2_e": 1.5,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [17]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

D_infection_medium = dict(default_medium)
D_infection_medium.update({
    "EX_o2_e": 2,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [18]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

E_infection_medium = dict(default_medium)
E_infection_medium.update({
    "EX_o2_e": 2.5,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

In [19]:
 # Make an infection condition medium dictionary with metabolites from SW's list and include the default medium conditions.

F_infection_medium = dict(default_medium)
F_infection_medium.update({
    "EX_o2_e": 3,
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0, # eliminate the glucose present in the default medium
})

## Infection Medium Gradient Study
### Function to run the gradient study

In [20]:
# Store the default, infected and uninfected medias in one dictionary
base_media = {
    "anaerobic infection": anaerobic_infection_medium,
    "0.5 oxygen infection": A_infection_medium,
    "1 oxygen infection medium": B_infection_medium, 
    "1.5 oxygen infection medium": C_infection_medium, 
    "2 oxygen infection medium": D_infection_medium,
    "2.5 oxygen infection medium": E_infection_medium, 
    "3 oxygen infection medium": F_infection_medium
}

In [29]:
import numpy as np
import pandas as pd

def run_medium_sweep(model_name, medium_condition, nutrient_rxn, nutrient_min, nutrient_max, step_level=0.5):
    """
    Run FBA across an adjustable nutrient gradient for one model and one medium condition.

    Parameters
    ----------
    model_name : str
        Name of the model to use, such as "control", "stm1090", or "mixed_stm2575".

    medium_condition : str
        Name of the base medium condition, such as "default" and "infection".

    step_level : float
        Nutrient increment size. Examples: 0.1, 0.5, 1, or 5.

    nutrient_min : float
        Starting nutrient uptake value. Default is 0.0.

    nutrient_max : float
        Maximum nutrient uptake value.

    nutrient_rxn : str
        Exchange reaction ID for the nutrient of interest. Example: "EX_o2_e" is the reaction ID for oxygen uptake.

    Returns
    -------
    pandas.DataFrame
        DataFrame with model name, medium condition, nutrient level, solver status,
        and objective value.
    """

    # Check that the requested medium exists
    if medium_condition not in base_media:
        raise ValueError(
            f"{medium_condition} is not in base_media. "
            f"Available media: {list(base_media.keys())}"
        )

    # Check that the requested model exists
    if model_name not in fba_models:
        raise ValueError(
            f"{model_name} is not in fba_models. "
            f"Available models: {list(fba_models.keys())}"
        )

    # Check that the nutrient step is valid
    if step_level <= 0:
        raise ValueError("step_level must be greater than 0.")

    # Build nutrient levels from nutrient_min to nutrient_max using the selected step size
    nutrient_levels = np.arange(nutrient_min, nutrient_max + step_level, step_level)

    # Remove values that go above nutrient_max
    nutrient_levels = nutrient_levels[nutrient_levels <= nutrient_max]

    # Round values to avoid floating-point artifacts like 0.30000000004
    nutrient_levels = np.round(nutrient_levels, 6) # why is this 6?

    # Make sure nutrient_max is included even if the step does not land on it exactly
    if nutrient_levels[-1] != nutrient_max:
        nutrient_levels = np.append(nutrient_levels, nutrient_max)

    # Get the selected base medium
    base_medium = base_media[medium_condition]

    # Get a fresh copy of the selected model
    model = fba_models[model_name].copy()

    results = []

    # Loop through each nutrient value
    for nutrient_value in nutrient_levels:

        # Copy the base medium so the original dictionary is not changed
        medium = base_medium.copy()

        # Set oxygen to the current oxygen value
        medium[nutrient_rxn] = float(nutrient_value)

        try:
            with model:
                model.medium = medium
                solution = model.optimize()

                if solution.status == "optimal":
                    objective_value = solution.objective_value
                else:
                    objective_value = np.nan

                results.append({
                    "model": model_name,
                    "medium_condition": medium_condition,
                    "nutrient": float(nutrient_value),
                    "status": solution.status,
                    "objective_value": objective_value
                })

        except Exception as error:
            results.append({
                "model": model_name,
                "medium_condition": medium_condition,
                "nutrient": float(nutrient_value),
                "status": "failed",
                "objective_value": np.nan,
                "error": str(error)
            })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values("nutrient").reset_index(drop=True)

    return results_df

# 08/05/26

I have a function that (1) build an array of step values in the minimum and maximum flux range defined in the parameters, (2) it makes a copy of the defined medium and creates a new medium for every number in the array, (3) runs FBA on the defined model with each medium. 

## Easy Version 

(stm_only_1090 and mixedstm1090_model) X (anaerobic infection medium) X (5 medium nutrients) X (10 step value) = 100 FBA runs 

Metbolites selected: 
EX_tet_e: 4.0
EX_no3_e: 0.5
EX_for_e: 4.0
EX_asp__L_e: 5.0
EX_galct__D_e: 10.0

```
stm1090_tet_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_tet_e", nutrient_min=0, nutrient_max=5, step_level=0.5)
mixedstm1090_tet_results = run_medium_sweep("mixedstm1090_model", "anaerobic infection", "EX_tet_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

stm1090_no3_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_no3_e", nutrient_min=0, nutrient_max=1, step_level=0.1)
mixedstm1090_no3_results = run_medium_sweep("mixedstm1090_model", "anaerobic infection", "EX_no3_e", nutrient_min=0, nutrient_max=1, step_level=0.1)

stm1090_for_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_for_e", nutrient_min=0, nutrient_max=5, step_level=0.5)
mixedstm1090_for_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_for_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

stm1090_asp_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_asp__L_e", nutrient_min=0, nutrient_max=5, step_level=0.5)
mixedstm1090_asp_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_asp__L_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

stm1090_galct_results = run_ medium_sweep("stm_only_1090", "anaerobic infection", "EX_galct__D_e", nutrient_min=0, nutrient_max=10, step_level=1)
mixedstm1090_galct_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_galct__D_e", nutrient_min=0, nutrient_max=10, step_level=1)
```

In [30]:
stm1090_tet_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_tet_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

Read LP format model from file /tmp/tmp7qnlmxnv.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


In [31]:
print(stm1090_tet_results)

            model     medium_condition  nutrient   status  objective_value
0   stm_only_1090  anaerobic infection       0.0  optimal         0.754956
1   stm_only_1090  anaerobic infection       0.5  optimal         0.774943
2   stm_only_1090  anaerobic infection       1.0  optimal         0.784873
3   stm_only_1090  anaerobic infection       1.5  optimal         0.794803
4   stm_only_1090  anaerobic infection       2.0  optimal         0.804733
5   stm_only_1090  anaerobic infection       2.5  optimal         0.814663
6   stm_only_1090  anaerobic infection       3.0  optimal         0.824593
7   stm_only_1090  anaerobic infection       3.5  optimal         0.834523
8   stm_only_1090  anaerobic infection       4.0  optimal         0.844452
9   stm_only_1090  anaerobic infection       4.5  optimal         0.854382
10  stm_only_1090  anaerobic infection       5.0  optimal         0.864312


stm1090_tet_results.to_csv(

In [35]:
# Mixed stm model, 10% threshold, anaerobic infection medium, tetrathionate gradient from 0 - 5 at 0.5 increments

mixedstm1090_tet_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_tet_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

Read LP format model from file /tmp/tmpougcbs13.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


In [36]:
print(mixedstm1090_tet_results)

             model     medium_condition  nutrient   status  objective_value
0   mixed_stm_1090  anaerobic infection       0.0  optimal         0.754225
1   mixed_stm_1090  anaerobic infection       0.5  optimal         0.775217
2   mixed_stm_1090  anaerobic infection       1.0  optimal         0.785147
3   mixed_stm_1090  anaerobic infection       1.5  optimal         0.795077
4   mixed_stm_1090  anaerobic infection       2.0  optimal         0.805008
5   mixed_stm_1090  anaerobic infection       2.5  optimal         0.814938
6   mixed_stm_1090  anaerobic infection       3.0  optimal         0.824868
7   mixed_stm_1090  anaerobic infection       3.5  optimal         0.834798
8   mixed_stm_1090  anaerobic infection       4.0  optimal         0.844728
9   mixed_stm_1090  anaerobic infection       4.5  optimal         0.854658
10  mixed_stm_1090  anaerobic infection       5.0  optimal         0.864588


In [37]:
# Stm only model, 10% threshold, anaerobic infection medium, Nitrate gradient from 0 - 1 at 0.1 increments

stm1090_no3_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_no3_e", nutrient_min=0, nutrient_max=1, step_level=0.1)

Read LP format model from file /tmp/tmpz6fg1fo_.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


In [38]:
print(stm1090_no3_results)

            model     medium_condition  nutrient   status  objective_value
0   stm_only_1090  anaerobic infection       0.0  optimal         0.820546
1   stm_only_1090  anaerobic infection       0.1  optimal         0.826579
2   stm_only_1090  anaerobic infection       0.2  optimal         0.831047
3   stm_only_1090  anaerobic infection       0.3  optimal         0.835516
4   stm_only_1090  anaerobic infection       0.4  optimal         0.839984
5   stm_only_1090  anaerobic infection       0.5  optimal         0.844452
6   stm_only_1090  anaerobic infection       0.6  optimal         0.848921
7   stm_only_1090  anaerobic infection       0.7  optimal         0.853389
8   stm_only_1090  anaerobic infection       0.8  optimal         0.857858
9   stm_only_1090  anaerobic infection       0.9  optimal         0.862326
10  stm_only_1090  anaerobic infection       1.0  optimal         0.866795


In [39]:
# Mixed Stm model, 10% threshold, anaerobic infection medium, Nitrate gradient from 0 - 1 at 0.1 increments

mixedstm1090_no3_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_no3_e", nutrient_min=0, nutrient_max=1, step_level=0.1)

Read LP format model from file /tmp/tmpndd33wcz.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


In [40]:
print(mixedstm1090_no3_results)

             model     medium_condition  nutrient   status  objective_value
0   mixed_stm_1090  anaerobic infection       0.0  optimal         0.820548
1   mixed_stm_1090  anaerobic infection       0.1  optimal         0.826854
2   mixed_stm_1090  anaerobic infection       0.2  optimal         0.831322
3   mixed_stm_1090  anaerobic infection       0.3  optimal         0.835791
4   mixed_stm_1090  anaerobic infection       0.4  optimal         0.840259
5   mixed_stm_1090  anaerobic infection       0.5  optimal         0.844728
6   mixed_stm_1090  anaerobic infection       0.6  optimal         0.849196
7   mixed_stm_1090  anaerobic infection       0.7  optimal         0.853665
8   mixed_stm_1090  anaerobic infection       0.8  optimal         0.858134
9   mixed_stm_1090  anaerobic infection       0.9  optimal         0.862602
10  mixed_stm_1090  anaerobic infection       1.0  optimal         0.867071


In [41]:
# Stm only model, 10% threshold, anaerobic infection medium, formate gradient from 0 - 5 at 0.5 increments

stm1090_for_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_for_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

Read LP format model from file /tmp/tmpgfyjxuqn.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


In [42]:
print(stm1090_for_results)

            model     medium_condition  nutrient   status  objective_value
0   stm_only_1090  anaerobic infection       0.0  optimal         0.844452
1   stm_only_1090  anaerobic infection       0.5  optimal         0.844452
2   stm_only_1090  anaerobic infection       1.0  optimal         0.844452
3   stm_only_1090  anaerobic infection       1.5  optimal         0.844452
4   stm_only_1090  anaerobic infection       2.0  optimal         0.844452
5   stm_only_1090  anaerobic infection       2.5  optimal         0.844452
6   stm_only_1090  anaerobic infection       3.0  optimal         0.844452
7   stm_only_1090  anaerobic infection       3.5  optimal         0.844452
8   stm_only_1090  anaerobic infection       4.0  optimal         0.844452
9   stm_only_1090  anaerobic infection       4.5  optimal         0.844452
10  stm_only_1090  anaerobic infection       5.0  optimal         0.844452


In [43]:
# Mixed Stm model, 10% threshold, anaerobic infection medium, formate gradient from 0 - 5 at 0.5 increments

mixedstm1090_for_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_for_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

Read LP format model from file /tmp/tmpqu7k10vn.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


In [44]:
print(mixedstm1090_for_results)

             model     medium_condition  nutrient   status  objective_value
0   mixed_stm_1090  anaerobic infection       0.0  optimal         0.844728
1   mixed_stm_1090  anaerobic infection       0.5  optimal         0.844728
2   mixed_stm_1090  anaerobic infection       1.0  optimal         0.844728
3   mixed_stm_1090  anaerobic infection       1.5  optimal         0.844728
4   mixed_stm_1090  anaerobic infection       2.0  optimal         0.844728
5   mixed_stm_1090  anaerobic infection       2.5  optimal         0.844728
6   mixed_stm_1090  anaerobic infection       3.0  optimal         0.844728
7   mixed_stm_1090  anaerobic infection       3.5  optimal         0.844728
8   mixed_stm_1090  anaerobic infection       4.0  optimal         0.844728
9   mixed_stm_1090  anaerobic infection       4.5  optimal         0.844728
10  mixed_stm_1090  anaerobic infection       5.0  optimal         0.844728


In [45]:
# Stm only model, 10% threshold, anaerobic infection medium, aspartate gradient from 0 - 5 at 0.5 increments

stm1090_asp_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_asp__L_e", nutrient_min=0, nutrient_max=5, step_level=0.5)


Read LP format model from file /tmp/tmp35kfc5q8.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


In [46]:
print(stm1090_asp_results)

            model     medium_condition  nutrient   status  objective_value
0   stm_only_1090  anaerobic infection       0.0  optimal         0.756995
1   stm_only_1090  anaerobic infection       0.5  optimal         0.766260
2   stm_only_1090  anaerobic infection       1.0  optimal         0.774943
3   stm_only_1090  anaerobic infection       1.5  optimal         0.783632
4   stm_only_1090  anaerobic infection       2.0  optimal         0.792320
5   stm_only_1090  anaerobic infection       2.5  optimal         0.801009
6   stm_only_1090  anaerobic infection       3.0  optimal         0.809698
7   stm_only_1090  anaerobic infection       3.5  optimal         0.818386
8   stm_only_1090  anaerobic infection       4.0  optimal         0.827075
9   stm_only_1090  anaerobic infection       4.5  optimal         0.835764
10  stm_only_1090  anaerobic infection       5.0  optimal         0.844452


In [50]:
# Mixed Stm model, 10% threshold, anaerobic infection medium, aspartate gradient from 0 - 5 at 0.5 increments

mixedstm1090_asp_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_asp__L_e", nutrient_min=0, nutrient_max=5, step_level=0.5)

Read LP format model from file /tmp/tmpjoyc7kvf.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


In [ ]:
print(mixedstm1090_asp_results)

In [ ]:
# Stm only model, 10% threshold, anaerobic infection medium, galactarate gradient from 0 - 10 at 1 increments

stm1090_galct_results = run_medium_sweep("stm_only_1090", "anaerobic infection", "EX_galct__D_e", nutrient_min=0, nutrient_max=10, step_level=1)

In [ ]:
print(stm1090_galct_results)

In [ ]:
# Mixed Stm only model, 10% threshold, anaerobic infection medium, galactarate gradient from 0 - 10 at 1 increments

mixedstm1090_galct_results = run_medium_sweep("mixed_stm_1090", "anaerobic infection", "EX_galct__D_e", nutrient_min=0, nutrient_max=10, step_level=1)

In [ ]:
print(mixedstm1090_galct_results)